# Preprocessing

In [1]:
DATASET = 'Amazon_Toys_and_Games'

REVIEWS_PATH = f'./{DATASET}.jsonl'
META_PATH = f'./meta_{DATASET}.jsonl'

INTER_SAVE_PATH = f'../{DATASET}/{DATASET}.inter'
ITEM_SAVE_PATH = f'../{DATASET}/{DATASET}.item'

ITEM_MAPPING_SAVE_PATH = f'../{DATASET}/item_mapping_{DATASET}.json'
USER_MAPPING_SAVE_PATH = f'../{DATASET}/user_mapping_{DATASET}.json'
ITEM_REVERSE_MAPPING_SAVE_PATH = f'../{DATASET}/item_reverse_mapping_{DATASET}.json'
USER_REVERSE_MAPPING_SAVE_PATH = f'../{DATASET}/user_reverse_mapping_{DATASET}.json'

DEGENERATE = True
MIN_USER_OCCURRENCE = 5
MIN_ITEM_OCCURRENCE = 5

In [2]:
import pandas as pd
import json
from tqdm import tqdm

### 1. Generation of .inter file

### 1.1 Reading the 'review' file

In [3]:
data = []
with open(REVIEWS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)

        data.append(
            {
                "user_id:token": record.get("user_id"),
                "item_id:token": record.get("parent_asin"),
                "rating:float": record.get("rating"),
                "timestamp:float": record.get("timestamp"),
            }
        )

df = pd.DataFrame(data)
df.head(5)

,user_id:token,item_id:token,rating:float,timestamp:float
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B09QH7QJS7,5.0,1677939664713
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B06XYKSKQP,3.0,1639855230760
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07XRSD5R9,5.0,1580949719154
3,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B007JWWUDW,5.0,1471542588000
4,AGKASBHYZPGTEPO6LWZPVJWB2BVA,B00MZG6OO8,3.0,1471541996000


### 1.2 Generate mappings

In [18]:
df["user_id:token"], user_index = pd.factorize(df["user_id:token"]) # eg. 42 -> "ASFAFASFASF"
df["item_id:token"], item_index = pd.factorize(df["item_id:token"])

reverse_item_index = {org_id: num for num, org_id in enumerate(item_index)} # eg. "ASFAFASFASF" -> 42
reverse_user_index = {org_id: num for num, org_id in enumerate(user_index)}

### 1.3 Filter entries with K-Core algorithm

In [4]:
while True:
    last_shape = df.shape

    df = df.groupby('user_id:token').filter(lambda x: len(x) >= MIN_USER_OCCURRENCE)
    df = df.groupby('item_id:token').filter(lambda x: len(x) >= MIN_USER_OCCURRENCE)

    if df.shape == last_shape:
        break

In [5]:
df.shape

(4005816, 4)

### 1.4 Validate the dataset

##### Check for null values

In [6]:
df.isna().sum()

user_id:token      0
item_id:token      0
rating:float       0
timestamp:float    0
dtype: int64

##### Check for invalid IDs

In [7]:
string_cols = df.select_dtypes(include="object").columns
for col in string_cols:
    print(col, "=> puste stringi:", (df[col] == "").sum())

user_id:token => puste stringi: 0
item_id:token => puste stringi: 0


##### Check for invalid types

In [ ]:
df[["rating:float", "timestamp:float"]].describe().map(lambda x: f"{x:,.0f}")

### 1.5 Save the .inter file

In [8]:
df.to_csv(INTER_SAVE_PATH, sep="\t", index=False)

### 1.6 Save the mappings to .json file

In [9]:
with open(ITEM_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(list(item_index), fp=f)

with open(ITEM_REVERSE_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(reverse_item_index, fp=f)

with open(USER_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(list(user_index), fp=f)

with open(USER_REVERSE_MAPPING_SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(reverse_user_index, fp=f)

NameError: name 'item_index' is not defined

In [ ]:
df.shape

### 1.7 Save valid items for .item file

In [10]:
valid_items = set(df['item_id:token'].values.tolist())

### 2. Generation of .item file

### 2.1 Reading the 'meta' file

In [11]:
def safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float('nan')

data = []
lines = 0
with open(META_PATH, 'r', encoding='utf-8') as f:
    lines = sum([1 for line in f])

data_items = []
data_titles = []
data_prices = []
data_brands = []
data_categories = []
data_sales_types = []
data_sales_ranks = []

with open(META_PATH, 'r', encoding='utf-8') as f:
    for line in tqdm(f, total=lines):
        record = json.loads(line)

        parent_asin = record.get("parent_asin")
        if parent_asin not in valid_items:
            continue

        # Cache relevant parts of the record
        details = record.get("details", {})
        best_sellers = details.get("Best Sellers Rank", {})

        # Handle Best Sellers Rank efficiently
        if best_sellers:
            max_key = max(best_sellers, key=best_sellers.get)
            max_value = best_sellers[max_key]
        else:
            max_key, max_value = None, float('nan')

        # Process price without try-except in loop
        price = record.get("price")
        try:
            price_float = float(price)
        except (TypeError, ValueError):
            price_float = float('nan')

        # Process categories without expensive string operations
        categories = record.get("categories", [])
        cat_str = "'" + "', '".join(categories) + "'" if categories else ''

        # Append data to lists
        data_items.append(reverse_item_index[parent_asin])
        data_titles.append(record.get("title"))
        data_prices.append(price_float)
        data_brands.append(record.get("store"))
        data_categories.append(cat_str)
        data_sales_types.append(max_key)
        data_sales_ranks.append(max_value)

# Create DataFrame from lists
df = pd.DataFrame({
    "item_id:token": data_items,
    "title:token": data_titles,
    "price:float": data_prices,
    "brand:token": data_brands,
    "categories:token_seq": data_categories,
    "sales_type:token": data_sales_types,
    "sales_rank:float": data_sales_ranks
})

df.head(5)

100%|██████████| 890874/890874 [00:11<00:00, 79653.33it/s]


,item_id:token,title:token,price:float,brand:token,categories:token_seq,sales_type:token,sales_rank:float


### 2.2 Validate the dataset

##### Check for null values

In [12]:
df.isna().sum()

item_id:token           0.0
title:token             0.0
price:float             0.0
brand:token             0.0
categories:token_seq    0.0
sales_type:token        0.0
sales_rank:float        0.0
dtype: float64

##### Check for invalid IDs

In [13]:
string_cols = df.select_dtypes(include="object").columns
for col in string_cols:
    print(col, "=> puste stringi:", (df[col] == "").sum())

##### Check for invalid types

In [13]:
df[["price:float", "sales_rank:float"]].describe().map(lambda x: f"{x:,.0f}")

,price:float,sales_rank:float
count,"484,502","624,142"
mean,57,"735,503"
std,162,"614,592"
min,0,1
25%,13,"267,574"
50%,24,"589,380"
75%,50,"1,054,191"
max,"22,000","13,516,189"


### 2.3 Save the .item file

In [14]:
df.to_csv(ITEM_SAVE_PATH, sep='\t', index=False, na_rep='')